# ДЗ 4. Olist: typed-схема, DML з PostgreSQL-фічами, JOIN, агрегати, set operations і window functions

**Студентка:** Samoilenko Mariia

**Датасет:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, CC BY-NC-SA 4.0).
Використано 6 CSV: customers, orders, order_items, products, sellers, order_reviews.

## Завдання 1. Setup і typed-схема Olist

In [4]:
import glob
import os
import subprocess
import sys


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("psycopg2-binary", "sqlalchemy", "pandas", "kagglehub", "fasteners", "platformdirs", "psutil")

# pgserver 0.1.4 має wheel-файли лише до cp312; його розширення зібране в abi3,
# тому на новішому Python (Colab) встановлюємо cp312-wheel з тегом abi3.
if sys.version_info < (3, 13):
    pip_install("pgserver==0.1.4")
else:
    wheel_dir = "/tmp/pgserver_wheel"
    subprocess.check_call([
        sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
        "--no-deps", "--only-binary=:all:", "--python-version", "3.12", "-d", wheel_dir,
    ])
    for whl in glob.glob(f"{wheel_dir}/pgserver-0.1.4-cp312-cp312-*.whl"):
        os.replace(whl, whl.replace("-cp312-cp312-", "-cp312-abi3-"))
    pip_install("--no-deps", *glob.glob(f"{wheel_dir}/pgserver-0.1.4-cp312-abi3-*.whl"))

In [5]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

pg = pgserver.get_server("/tmp/hw4_pg", cleanup_mode="stop")
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def q(sql_text):
    """SELECT → DataFrame."""
    return pd.read_sql(text(sql_text), engine)


def run(sql_text):
    """DDL / DML без результату в одній транзакції."""
    with engine.begin() as conn:
        conn.execute(text(sql_text))


def dml(sql_text):
    """DML з RETURNING в одній транзакції → DataFrame повернутих рядків."""
    with engine.begin() as conn:
        result = conn.execute(text(sql_text))
        return pd.DataFrame(result.fetchall(), columns=list(result.keys()))

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### Отримання CSV

Варіант A — KaggleHub (публічний датасет завантажується і без облікових даних).
Варіант B — копія 6 потрібних CSV у репозиторії (`data/olist/`), якщо KaggleHub недоступний.

In [6]:
FILES = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}
REPO_RAW = "https://raw.githubusercontent.com/mashamm/goit-rdb-hw-04/main/data/olist"

try:
    import kagglehub
    dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print("Варіант A (KaggleHub):", dataset_dir)
except Exception as exc:
    print("KaggleHub недоступний:", exc)
    dataset_dir = os.path.abspath("../data/olist")
    if not all(os.path.exists(os.path.join(dataset_dir, f)) for f in FILES.values()):
        from urllib.request import urlretrieve
        dataset_dir = "/content/data/olist" if os.path.isdir("/content") else os.path.abspath("data/olist")
        os.makedirs(dataset_dir, exist_ok=True)
        for f in FILES.values():
            urlretrieve(f"{REPO_RAW}/{f}", os.path.join(dataset_dir, f))
    print("Варіант B (CSV з репозиторію):", dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Варіант A (KaggleHub): /kaggle/input/brazilian-ecommerce


### Raw-шар: CSV → `*_raw` таблиці (проміжний шар без constraints)

In [7]:
run("DROP TABLE IF EXISTS olist_customers CASCADE; DROP TABLE IF EXISTS olist_orders CASCADE;")

raw_counts = {}
for table_name, file_name in FILES.items():
    df = pd.read_csv(os.path.join(dataset_dir, file_name))
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    raw_counts[table_name] = len(df)
    print(f"{table_name:26s} {df.shape}")

olist_customers_raw        (99441, 5)
olist_orders_raw           (99441, 8)
olist_order_items_raw      (112650, 7)
olist_products_raw         (32951, 9)
olist_sellers_raw          (3095, 4)
olist_order_reviews_raw    (99224, 7)


### Typed-схема

In [8]:
run("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
""")
print("OK")

OK


### Raw → typed через `INSERT ... SELECT`

У сирому CSV колонки названі з помилкою `product_name_lenght` / `product_description_lenght`; у typed-схемі вони нормалізовані.

In [9]:
run("""
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER,
       customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP,
       order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP,
       order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER, product_description_lenght::INTEGER,
       product_photos_qty::INTEGER, product_weight_g::INTEGER,
       product_length_cm::INTEGER, product_height_cm::INTEGER, product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP, price::NUMERIC(12, 2), freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id, order_id, review_score, review_comment_title,
    review_comment_message, review_creation_date, review_answer_timestamp
)
SELECT review_id, order_id, review_score::SMALLINT, review_comment_title,
       review_comment_message, review_creation_date::TIMESTAMP, review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")
print("OK")

OK


### Smoke-test: typed vs raw

In [10]:
typed = q("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name
""")
typed["raw_rows"] = typed["table_name"].map({
    "customers": raw_counts["olist_customers_raw"],
    "orders": raw_counts["olist_orders_raw"],
    "order_items": raw_counts["olist_order_items_raw"],
    "products": raw_counts["olist_products_raw"],
    "sellers": raw_counts["olist_sellers_raw"],
    "reviews": raw_counts["olist_order_reviews_raw"],
})
typed["lost_rows"] = typed["raw_rows"] - typed["n_rows"]
typed

,table_name,n_rows,raw_rows,lost_rows
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,reviews,99224,99224,0
5,sellers,3095,3095,0


**Висновок.** Кількість рядків у всіх typed-таблицях дорівнює raw: 99 441 customers і orders, 112 650 order_items, 32 951 products, 3 095 sellers, 99 224 reviews, втрат 0. Усі значення коректно привелися до типів, а всі FK знайшли батьківський рядок: кожне замовлення має клієнта, кожна позиція — товар і продавця, кожен відгук — замовлення. `CHECK`-обмеження (статус, невід'ємні ціни, `review_score` 1–5) теж жодного рядка не відкинули.

## Завдання 2. DML-запити з PostgreSQL-фічами

### 2.1. `INSERT ... RETURNING`: alert для продавця з найбільшою кількістю запізнень

In [11]:
run("""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")
print("OK")

OK


In [12]:
res = dml("""
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT so.seller_id, COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT seller_id, 'late_delivery', 4,
       'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 1


,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-09-28 06:20:26.695106+00:00


**Що змінює запит.** Він вставляє один alert типу `late_delivery` для продавця з найбільшою кількістю запізнених доставок: 172 замовлення продавця `4a3ca9315b...`. `RETURNING` одразу повертає згенерований `alert_id`, `created_at` і текст деталей. Не потрібен окремий `SELECT` після вставки, а результат можна зберегти як audit-log: що саме, коли і з яким ідентифікатором створив ETL-крок.

### 2.2. `INSERT ... ON CONFLICT DO UPDATE`: idempotent upsert `seller_score`

In [13]:
run("""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")
print("OK")

OK


In [14]:
res = dml("""
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT so.seller_id,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
           COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT seller_id, avg_rating, n_reviews,
       CASE
           WHEN avg_rating IS NULL THEN NULL
           WHEN avg_rating >= 4.5 THEN 'gold'
           WHEN avg_rating >= 3.5 THEN 'silver'
           ELSE 'bronze'
       END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-09-28 06:20:26.961741+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-09-28 06:20:26.961741+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-09-28 06:20:26.961741+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-09-28 06:20:26.961741+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-09-28 06:20:26.961741+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-09-28 06:20:26.961741+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-09-28 06:20:26.961741+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-09-28 06:20:26.961741+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-09-28 06:20:26.961741+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-09-28 06:20:26.961741+00:00


**Перевірка ідемпотентності:** запускаємо той самий upsert удруге — кількість рядків у `seller_score` не змінюється.

In [15]:
before = q("SELECT COUNT(*) AS n FROM seller_score")["n"][0]
res = dml("""WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT so.seller_id,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
           COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT seller_id, avg_rating, n_reviews,
       CASE
           WHEN avg_rating IS NULL THEN NULL
           WHEN avg_rating >= 4.5 THEN 'gold'
           WHEN avg_rating >= 3.5 THEN 'silver'
           ELSE 'bronze'
       END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at""")
after = q("SELECT COUNT(*) AS n FROM seller_score")["n"][0]
print(f"рядків до: {before}, після повторного запуску: {after}, оновлено через ON CONFLICT: {len(res)}")
assert before == after
q("""
SELECT COALESCE(tier, '(no reviews)') AS tier, COUNT(*) AS n_sellers,
       ROUND(AVG(avg_rating), 2) AS avg_rating, SUM(n_reviews) AS reviews
FROM seller_score
GROUP BY tier
ORDER BY n_sellers DESC
""")

рядків до: 3095, після повторного запуску: 3095, оновлено через ON CONFLICT: 3095


,tier,n_sellers,avg_rating,reviews
0,silver,1568,4.06,87123
1,gold,983,4.83,7093
2,bronze,539,2.30,5031
3,(no reviews),5,NaN,0


**Пояснення.**

- **`SELECT DISTINCT` у `seller_orders`** переводить дані з грануляності «позиція замовлення» на «продавець × замовлення». Якщо продавець продав у замовленні 3 позиції, замовлення враховується один раз, а не тричі.
- **Review агрегуються на рівні `order_id`** (`review_per_order`), бо в Olist 547 замовлень мають кілька відгуків. Без попередньої агрегації join дав би кілька рядків на замовлення, і оцінка множилася б і через позиції, і через дублікати відгуків.
- **`ON CONFLICT (seller_id) DO UPDATE`**: `seller_id` — PK, тому повторний запуск не створює нових рядків, а оновлює наявні. Перевірка вище: 3 095 рядків до й після повторного запуску.
- **Продавці без review** отримують `avg_rating = NULL`, `n_reviews = 0` і `tier = NULL`: `LEFT JOIN` зберігає продавця, `AVG` від порожнього набору дає NULL, `COUNT(rpo.order_id)` рахує лише наявні відгуки. Таких продавців 5 — чесно показати, що рейтингу немає, краще, ніж підставити 0.

**Результат:** 1 568 продавців silver (середній рейтинг 4,06), 983 gold (4,83), 539 bronze (2,30).

**Обмеження датасету:** review прив'язаний до замовлення, а не до продавця. У 1 278 замовленнях кілька продавців, і відгук такого замовлення зараховується кожному з них, навіть якщо проблема була лише в одного. Це прийнятне навчальне спрощення.

### 2.3. `UPDATE ... FROM`: прапорець запізнення `is_late`

In [16]:
run("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
""")
print("OK")

OK


In [17]:
res = dml("""
WITH delivery_flags AS (
    SELECT order_id,
           (order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 99441


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


In [18]:
q("""
SELECT is_late, COUNT(*) AS n_orders, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM olist_orders
GROUP BY is_late
ORDER BY is_late
""")

,is_late,n_orders,pct
0,False,92906,93.43
1,True,6535,6.57


**Що змінює запит.** `UPDATE ... FROM` встановлює `is_late` для всіх 99 441 замовлення за поточними датами доставки: 6 535 замовлень (6,57 %) доставлено пізніше оцінної дати. Запит ідемпотентний: він не «дописує TRUE», а щоразу перераховує значення з дат. Тому повторний запуск дає той самий результат, а якщо дати оновляться, прапорець теж виправиться. `RETURNING` повертає змінені рядки для контролю.

### 2.4. `DELETE ... RETURNING`: видалення застарілих alert-ів

In [19]:
res = dml("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 0


,alert_id,seller_id,alert_type,created_at


**Результат порожній, і це очікувано:** єдиний alert створено кілька секунд тому, тож жоден запис не старший за 30 днів. `DELETE ... RETURNING` не падає на порожньому наборі, а явно показує, що нічого не видалено.

Щоб продемонструвати, що механізм працює, додаємо тестовий alert із `created_at` на 45 днів у минулому й повторюємо той самий `DELETE`:

In [20]:
res = dml("""
INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
SELECT seller_id, 'data_quality', 2, 'Test alert backdated by 45 days', NOW() - INTERVAL '45 days'
FROM seller_score
ORDER BY seller_id
LIMIT 1
RETURNING alert_id, seller_id, alert_type, created_at
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 1


,alert_id,seller_id,alert_type,created_at
0,2,0015a82c2db000af6aaaf3ae2ecb0532,data_quality,2026-08-14 06:20:30.213991+00:00


In [21]:
res = dml("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 1


,alert_id,seller_id,alert_type,created_at
0,2,0015a82c2db000af6aaaf3ae2ecb0532,data_quality,2026-08-14 06:20:30.213991+00:00


**Висновок.** Другий `DELETE` видалив рівно тестовий alert (`alert_id = 2`, тип `data_quality`), а `RETURNING` повернув його. Цей рядок можна записати в audit-таблицю або журнал — видно, що саме видалено. Alert `late_delivery` з кроку 2.1 залишився, бо він свіжий.

## Завдання 3. `customer_segments`

### Дедуплікований customer dimension

In [22]:
run("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (customer_unique_id, latest_customer_id, customer_state, customer_city)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id, c.customer_id, c.customer_state, c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o ON o.customer_id = c.customer_id
ORDER BY c.customer_unique_id, o.order_purchase_timestamp DESC NULLS LAST, c.customer_id;
""")
print("OK")

OK


In [23]:
q("""
SELECT
    (SELECT COUNT(*) FROM olist_customers)    AS customer_ids,
    (SELECT COUNT(*) FROM olist_customer_dim) AS unique_customers
""")

,customer_ids,unique_customers
0,99441,96096


### DDL і idempotent-наповнення

In [24]:
run("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),
    rfm_score           SMALLINT NOT NULL CHECK (rfm_score BETWEEN 1 AND 5),
    recency_days        INTEGER CHECK (recency_days IS NULL OR recency_days >= 0),
    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0 CHECK (monetary_value >= 0),
    n_orders            INTEGER NOT NULL DEFAULT 0 CHECK (n_orders >= 0),
    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")
print("OK")

OK


In [25]:
res = dml("""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT o.order_id, o.customer_id, o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
           MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT cs.customer_unique_id, cs.n_orders, cs.monetary_value,
           CASE WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER END AS recency_days,
           CASE
               WHEN cs.monetary_value >= 1000 THEN 'VIP'
               WHEN cs.last_order_date IS NOT NULL
                    AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                    AND cs.n_orders = 1 THEN 'new'
               WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
               ELSE 'inactive'
           END AS segment_name,
           CASE
               WHEN cs.monetary_value >= 1000 THEN 5
               WHEN cs.monetary_value >= 500  THEN 4
               WHEN cs.n_orders >= 2          THEN 3
               WHEN cs.n_orders = 1           THEN 2
               ELSE 1
           END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
""")
print("RETURNING rows:", len(res))
res.head(10)

RETURNING rows: 96096


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


Повторний запуск (idempotency) і розподіл сегментів:

In [26]:
res = dml("""WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT o.order_id, o.customer_id, o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
           MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT cs.customer_unique_id, cs.n_orders, cs.monetary_value,
           CASE WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER END AS recency_days,
           CASE
               WHEN cs.monetary_value >= 1000 THEN 'VIP'
               WHEN cs.last_order_date IS NOT NULL
                    AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                    AND cs.n_orders = 1 THEN 'new'
               WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
               ELSE 'inactive'
           END AS segment_name,
           CASE
               WHEN cs.monetary_value >= 1000 THEN 5
               WHEN cs.monetary_value >= 500  THEN 4
               WHEN cs.n_orders >= 2          THEN 3
               WHEN cs.n_orders = 1           THEN 2
               ELSE 1
           END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders""")
print("повторний upsert оновив рядків:", len(res))
q("""
SELECT segment_name, COUNT(*) AS n_customers,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct,
       ROUND(AVG(monetary_value), 2) AS avg_monetary,
       ROUND(AVG(n_orders), 2) AS avg_orders
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC
""")

повторний upsert оновив рядків: 96096


,segment_name,n_customers,pct,avg_monetary,avg_orders
0,regular,45982,47.85,224.57,1.07
1,inactive,40032,41.66,59.28,1.00
2,new,8891,9.25,140.39,1.00
3,VIP,1191,1.24,1591.81,1.11


Перевірка constraints:

In [27]:
q("""
SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'::text, 'new'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_uniq...


**Пояснення.**

- **Чому `customer_unique_id`.** У Olist `customer_id` створюється для кожного замовлення. 99 441 `customer_id` відповідають лише 96 096 реальним покупцям, і 2 997 з них мають кілька `customer_id`. За `customer_id` frequency завжди була б 1, і повторних покупців не було б видно.
- **Навіщо `olist_customer_dim`.** FK має посилатися на унікальний ключ. В `olist_customers` `customer_unique_id` не унікальний, тому потрібна дедуплікована таблиця-вимір, де він є PK. `DISTINCT ON` з сортуванням за датою замовлення вибирає найсвіжіший `customer_id` покупця.
- **Чому `reference_date` з датасету.** Останнє замовлення в Olist датоване 2018 роком, тож `reference_date = 2018-10-18`. З `CURRENT_DATE` усі покупці мали б recency у тисячі днів і всі стали б «inactive», а результат змінювався б щодня. Дата з даних робить сегментацію стабільною й відтворюваною.
- **Як `ON CONFLICT` робить сегментацію повторюваною.** Завдяки `UNIQUE (customer_unique_id)` повторний запуск оновлює сегмент наявного покупця й `assigned_at`, а не додає дублікат. Повторний upsert оновив ті самі 96 096 рядків.

**Розподіл:** regular 47,9 %, inactive 41,7 %, new 9,3 %, VIP 1,2 % (середні витрати 1 591,81). Навіть VIP у середньому мають лише 1,11 замовлення, тобто повторні покупки в Olist рідкісні.

## Завдання 4. JOIN-запити з business-interpretation

### 4.1. INNER JOIN (4 таблиці)

**Business-question:** Які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [28]:
q("""
SELECT p.product_category_name, s.seller_state,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders   AS o ON o.order_id   = oi.order_id
JOIN olist_products AS p ON p.product_id = oi.product_id
JOIN olist_sellers  AS s ON s.seller_id  = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10
""")

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** Усі 10 найбільших комбінацій «категорія × штат продавця» припадають на Сан-Паулу (SP). Лідирують `cama_mesa_banho` (постільна й банна білизна, 1,07 млн BRL, 8 214 замовлень) і `relogios_presentes` (годинники й подарунки, 1,02 млн BRL). У годинників удвічі менше замовлень, але приблизно такий самий revenue — у них вищий середній чек. INNER JOIN відкидає позиції без товару, продавця чи недоставлених замовлень, тому це revenue саме доставлених замовлень.

### 4.2. LEFT JOIN + COALESCE

**Business-question:** Які customer-level features можна побудувати для реального покупця на рівні customer_unique_id?

In [29]:
q("""
WITH order_totals AS (
    SELECT o.order_id, c.customer_unique_id,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_unique_id
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT ot.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id, cd.customer_state,
       COALESCE(cs.n_orders, 0) AS n_orders,
       COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
       COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20
""")

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


**Interpretation:** Отримано ознаки для кожного реального покупця: кількість замовлень, сумарні витрати й середня оцінка. Найбільші витрати — 13 664 BRL за одне замовлення (оцінка 1). Серед топ-20 є покупці з 2 і 4 замовленнями. CTE `review_per_order` спершу усереднює відгуки до рівня замовлення, тому оцінка не множиться на кількість позицій. `COALESCE(..., 0)` для `avg_review_score` дає 0 покупцям без відгуків (наприклад, рядок з оцінкою 0.0). Для ML це небезпечно, бо 0 виглядає як «найгірша оцінка»; краще залишати NULL і додавати прапорець `has_review`.

### 4.3. FULL OUTER JOIN

**Business-question:** Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [30]:
q("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT COALESCE(c.state, s.state) AS state, c.n_customers, s.n_sellers,
       CASE
           WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
           WHEN c.state IS NOT NULL THEN 'customers_only'
           ELSE 'sellers_only'
       END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s ON s.state = c.state
ORDER BY state_presence DESC, state
""")

,state,n_customers,n_sellers,state_presence
0,AL,413,NaN,customers_only
1,AP,68,NaN,customers_only
2,RR,46,NaN,customers_only
3,TO,280,NaN,customers_only
4,AC,81,1.0,both
5,AM,148,1.0,both
6,BA,3380,19.0,both
7,CE,1336,13.0,both
8,DF,2140,30.0,both
9,ES,2033,23.0,both


**Interpretation:** Клієнти є в усіх 27 штатах, продавці — лише у 23. Штати AL, AP, RR і TO мають лише клієнтів (`customers_only`), а штатів лише з продавцями немає. Ринок сильно концентрований: у SP 41 746 `customer_id` і 1 849 продавців. У штатах AC, AM, MA, PA, PI лише по одному продавцю — там покупці майже повністю залежать від доставки з інших штатів.

### 4.4. SELF JOIN

**Business-question:** Які пари продавців з одного штату, але з різних міст, можуть бути кандидатами для регіонального порівняння — і скільки таких пар у кожному штаті?

In [31]:
q("""
SELECT s1.seller_state,
       COUNT(*) AS n_pairs_other_city,
       MIN(s1.seller_id || ' / ' || s2.seller_id) AS example_pair
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
  ON s1.seller_state = s2.seller_state
 AND s1.seller_id < s2.seller_id
 AND s1.seller_city IS DISTINCT FROM s2.seller_city
GROUP BY s1.seller_state
ORDER BY n_pairs_other_city DESC
LIMIT 10
""")

,seller_state,n_pairs_other_city,example_pair
0,SP,1457751,0015a82c2db000af6aaaf3ae2ecb0532 / 002100f778ceb8431b7a1020ff7ab48f
1,PR,51665,00fc707aaaad2d31347cf883cd2dfe10 / 01266d4c46afa519678d16a8b683d325
2,MG,27184,00d8b143d12632bad99c0ad66ad52825 / 014c0679dd340a0e338872e7ec85666a
3,SC,17208,034b9387752bfa309f350a70cd45ffa3 / 0417b067eeab773d2f7061a726dc477f
4,RJ,10151,001e6ad469a905060d959994f1b41e4f / 02a2272692e13558373c66db98f05e2e
5,RS,7698,04843805947f0fc584fc1969b6e50fe7 / 04ee0ec01589969663ba5967c0e0bdc0
6,GO,519,003554e2dce176b5555353e4f3555ac8 / 01bcc9d254a0143f0ce9791b960b2a47
7,ES,231,001cca7ae9ae17fb1caed9dfb1094831 / 15aac934c58d886785ac1b17953ea898
8,BA,149,1444c08e64d55fb3c25f0f09c07ffcf2 / 2b402d5dc42554061f8ea98d1916f148
9,DF,57,191d55f7edb8524d525ed01b20fe32e8 / 3c487ae8f8d7542beff5788e2e0aea83


**Interpretation:** Self join з'єднує таблицю продавців саму з собою за штатом, а `s1.seller_id < s2.seller_id` прибирає дзеркальні пари й пару продавця із самим собою. У SP понад 1,46 млн пар продавців з різних міст, у PR — 51 665, у MG — 27 184. Порівнювати всі пари недоцільно, тож для регіонального аналізу пари варто фільтрувати за категорією або обсягом продажів. Рядок `example_pair` показує конкретну пару-кандидат для кожного штату.

### 4.5. Anti-join (LEFT JOIN ... IS NULL)

**Business-question:** Які замовлення не мають жодного review-запису і в яких вони статусах?

In [32]:
q("""
SELECT o.order_status,
       COUNT(*) AS orders_without_review,
       MIN(o.order_purchase_timestamp) AS first_purchase,
       MAX(o.order_purchase_timestamp) AS last_purchase
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
GROUP BY o.order_status
ORDER BY orders_without_review DESC
""")

,order_status,orders_without_review,first_purchase,last_purchase
0,delivered,646,2016-10-05 20:20:59,2018-08-27 17:01:43
1,shipped,75,2016-10-05 16:57:30,2018-08-07 21:40:46
2,canceled,20,2017-02-12 13:41:41,2018-09-06 10:48:12
3,unavailable,14,2017-02-06 08:18:30,2018-08-11 21:38:00
4,processing,6,2017-05-27 17:48:11,2017-12-28 12:46:11
5,invoiced,5,2017-04-24 13:37:12,2018-05-16 14:57:49
6,created,2,2017-12-05 01:07:58,2018-02-09 17:21:04


**Interpretation:** 768 замовлень не мають жодного відгуку, найбільше — 646 доставлених. Отже, відсутність відгуку не означає проблем із замовленням: покупець просто не відповів. Скасовані (20) і недоступні (14) замовлення без відгуку очікувані. Для моделі задоволеності такі замовлення не можна вважати «нейтральними»: у них немає мітки.

### 4.6. Anti-join (NOT EXISTS)

**Business-question:** Скільки товарів у каталозі жодного разу не продавалися?

In [33]:
q("""
SELECT COUNT(*) AS never_sold_products,
       (SELECT COUNT(*) FROM olist_products) AS all_products
FROM olist_products AS p
WHERE NOT EXISTS (
    SELECT 1 FROM olist_order_items AS oi WHERE oi.product_id = p.product_id
)
""")

,never_sold_products,all_products
0,0,32951


**Interpretation:** Усі 32 951 товар каталогу продавалися хоча б раз: `NOT EXISTS` не знайшов товарів без позицій. Це очікувано, бо таблицю products в Olist сформовано з товарів, які фігурують у замовленнях, тобто це не повний каталог. Тому на цих даних не можна аналізувати товари, що не продаються.

## Завдання 5. GROUP BY, HAVING і агрегати

### 5.1. Revenue per category (WHERE + HAVING)

**Business-question:** Які категорії мають понад 100 000 BRL gross revenue у доставлених замовленнях?

In [34]:
q("""
SELECT p.product_category_name,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
       SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
       AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p ON p.product_id = oi.product_id
JOIN olist_orders   AS o ON o.order_id   = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 100000
ORDER BY gross_revenue DESC NULLS LAST
""")

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8647,1233131.72,178957.81,130.28
1,relogios_presentes,5495,1166176.98,98156.14,199.04
2,cama_mesa_banho,9272,1023434.76,201774.50,93.44
3,esporte_lazer,7530,954852.55,163404.36,113.25
4,informatica_acessorios,6530,888724.61,143999.16,116.26
5,moveis_decoracao,6307,711927.69,168402.23,87.25
6,utilidades_domesticas,5743,615628.69,142763.56,90.60
7,cool_stuff,3559,610204.10,81476.79,164.12
8,automotivo,3810,578966.65,90488.10,139.85
9,brinquedos,3804,471286.48,75774.58,116.94


**Interpretation:** Понад 100 000 BRL у доставлених замовленнях заробили багато категорій; лідери — `beleza_saude` (1,23 млн BRL), `relogios_presentes` (1,17 млн) і `cama_mesa_banho` (1,02 млн). Категорія `pcs` має лише 177 замовлень, але найвищу середню ціну позиції (1 098,92 BRL), тож вона потрапила у список за рахунок чека, а не обсягу. У `cama_mesa_banho` доставка становить майже 20 % від ціни товарів, тож логістика для неї особливо важлива.

**WHERE vs HAVING на цьому запиті.** `WHERE o.order_status = 'delivered'` фільтрує **рядки до агрегації**: у суму не потрапляють позиції скасованих чи недоставлених замовлень. `HAVING SUM(oi.price) > 100000` фільтрує **групи після агрегації**: категорія залишається в результаті, лише якщо її підсумковий revenue перевищує поріг. Перенести умову `SUM(...) > 100000` у `WHERE` неможливо, бо на етапі `WHERE` суми ще не обчислено.

### 5.2. AOV per customer-state (HAVING)

**Business-question:** Який середній чек (AOV) у штатах, де є щонайменше 100 замовлень?

In [41]:
q("""
WITH order_totals AS (
    SELECT o.order_id, c.customer_state,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_state
)
SELECT customer_state,
       COUNT(*) AS n_orders,
       SUM(order_total)::NUMERIC(12, 2) AS gmv,
       (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST
""")

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Interpretation:** `HAVING COUNT(*) >= 100` залишає 24 штати з достатньою кількістю замовлень. Найвищий середній чек — на північному сході (PB 265,01 BRL, AL 234,13, RO 233,03), а найнижчий — у SP (143,12 BRL), хоча SP дає найбільший GMV (5,92 млн BRL). Ймовірно, у віддалених штатах вищий чек через дорожчу доставку й більші замовлення. Штат клієнта — корисна ознака для прогнозу AOV.

### 5.3. STRING_AGG (HAVING)

**Business-question:** Які категорії купували повторні покупці (≥ 2 замовлення) з найбільшими витратами?

In [42]:
q("""
SELECT c.customer_unique_id,
       COUNT(DISTINCT o.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
       STRING_AGG(DISTINCT COALESCE(p.product_category_name, '(unknown)'), ', '
                  ORDER BY COALESCE(p.product_category_name, '(unknown)')) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 15
""")

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


**Interpretation:** `HAVING COUNT(DISTINCT o.order_id) >= 2` залишає лише повторних покупців. Найбільше витратив покупець із 2 замовленнями в категорії `eletroportateis` (7 388 BRL). Більшість топових повторних покупців купують в одній категорії (`pcs`, `telefonia`, `informatica_acessorios`), і лише деякі поєднують дві, як-от `moveis_sala, utilidades_domesticas`. `STRING_AGG(DISTINCT ...)` дає компактний профіль інтересів покупця, який можна використати для рекомендацій.

## Завдання 6. Set operations, window function і Reflection

### 6.1. UNION

**Business-question:** Скільки унікальних штатів представлено серед клієнтів або продавців?

In [43]:
q("""
SELECT COUNT(*) AS n_states_any_role
FROM (
    SELECT customer_state AS state FROM olist_customers
    UNION
    SELECT seller_state FROM olist_sellers
) AS all_states
""")

,n_states_any_role
0,27


**Interpretation:** Разом клієнти й продавці представлені у 27 штатах, тобто у всіх штатах Бразилії з федеральним округом. Обрано `UNION`, а не `UNION ALL`, бо потрібен **перелік унікальних** штатів: `UNION` прибирає дублікати, тоді як `UNION ALL` повернув би понад 100 тисяч рядків (по одному на кожного клієнта й продавця).

### 6.2. INTERSECT / EXCEPT

**Business-question:** У яких штатах є і клієнти, і продавці, а в яких — лише клієнти?

In [44]:
q("""
SELECT state, 'both' AS presence
FROM (
    SELECT customer_state AS state FROM olist_customers
    INTERSECT
    SELECT seller_state FROM olist_sellers
) AS both_roles
UNION ALL
SELECT state, 'customers_only'
FROM (
    SELECT customer_state AS state FROM olist_customers
    EXCEPT
    SELECT seller_state FROM olist_sellers
) AS customers_only
ORDER BY presence, state
""")

,state,presence
0,AC,both
1,AM,both
2,BA,both
3,CE,both
4,DF,both
5,ES,both
6,GO,both
7,MA,both
8,MG,both
9,MS,both


**Interpretation:** `INTERSECT` повертає 23 штати, де є і клієнти, і продавці, а `EXCEPT` — 4 штати лише з клієнтами (AL, AP, RR, TO). Результат збігається з FULL OUTER JOIN із 4.3, але set-оператори прямо формулюють питання «спільні» й «різниця». Результати з'єднано через `UNION ALL`, бо обидві частини вже не перетинаються й дедуплікація не потрібна.

### 6.3. Window: ROW_NUMBER()

**Business-question:** Яке найновіше замовлення кожного реального покупця і які покупці роблять найбільше повторних замовлень?

In [45]:
q("""
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.customer_id, o.order_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (
               PARTITION BY c.customer_unique_id
               ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
           ) AS rn,
           COUNT(*) OVER (PARTITION BY c.customer_unique_id) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
)
SELECT customer_unique_id, customer_id, order_id, order_purchase_timestamp, n_orders
FROM ranked_orders
WHERE rn = 1
ORDER BY n_orders DESC, customer_unique_id
LIMIT 15
""")

,customer_unique_id,customer_id,order_id,order_purchase_timestamp,n_orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,a682769c4bc10fc6ef2101337a6c83c9,d3582fd5ccccd9cb229a63dfb417c86f,2018-08-20 19:14:26,17
1,3e43e6105506432c953e165fb2acf44c,1815fdaaae692479110428598596b5e5,ff89ef7b3952bba5ac06d61c4a79ffbe,2018-02-27 18:36:39,9
2,1b6c7548a2a1f9037c1fd3ddfed95f33,fff7466a253c0e59499ea943462c10f9,43f08913407cac4e74a898d968e58c1a,2018-02-14 13:22:12,7
3,6469f99c1f9dfae7733b25662e7f1782,4478700ff008d3fffa02e95d89b9795f,ccb1046449f1a494de800567e7cb5979,2018-06-28 00:43:34,7
4,ca77025e7201e3b30c44b472ff346268,6ccedfba5919d72fcc8c51bfa982de62,8dacd1da7c4dc40d5c59e7ddd93b3ac8,2018-06-01 11:38:29,7
5,12f5d6e1cbf93dafd9dcc19095df0b3d,ed1793d2d1e4175d5846ce7ebb4a01f5,747996a66f5aa711deb8ae58f5ae46a0,2017-01-05 15:25:10,6
6,47c1a3033b8b77b3ab6e109eb4d5fdf3,5daf8c734e89f88e5305f9283e7ca135,b86ec88f5bffbf1f61fb5daed332e355,2018-01-24 15:15:26,6
7,63cfc61cee11cbe306bff5857d00bfe4,906614004e38a1e922ff033563bfdcf2,c29713133e5e33937af1a6e1e1370552,2018-05-28 17:20:02,6
8,dc813062e0fc23409cd255f7f53c7074,4fd0b9bb860c9a3d841746e1cb742314,f0031384d36ff045ee1e866942d0fa7a,2018-08-23 00:07:26,6
9,de34b16117594161a6a89c50b289d35a,cbbd508281ea66e2f36b4acfc28a85b0,d29eca3633b0b60adf268cae9133cd93,2018-01-17 23:34:21,6


**Interpretation:** `ROW_NUMBER()` у межах `customer_unique_id` нумерує замовлення від найновішого, і `rn = 1` залишає останнє замовлення кожного покупця. `COUNT(*) OVER` без згортання рядків додає загальну кількість замовлень. Найактивніший покупець має 17 замовлень (останнє — 2018-08-20), ще кілька покупців мають 6–9 замовлень. Такий патерн «останній запис на сутність» — стандартний крок побудови ознак, наприклад recency.

## Reflection

**Найскладніший JOIN** — не з погляду синтаксису, а з погляду грануляності: LEFT JOIN відгуків до `order_items` непомітно множить оцінки. Доводилося щоразу перевіряти, на якому рівні (позиція, замовлення, продавець, покупець) агрегуються дані. Найскладніше було сформулювати бізнес-питання для SELF JOIN: сам по собі перелік пар продавців мало що дає, тож я звела його до кількості пар-кандидатів за штатом.

**Найкорисніша фіча для production-ETL** — `ON CONFLICT`: вона робить завантаження ідемпотентним, і після збою крок можна просто запустити знову. `RETURNING` доповнює її дешевим audit-log.

**Window functions** я використала б для останнього замовлення покупця, інтервалів між покупками (`LAG`), ковзних середніх оцінок продавця та ранжування товарів у категорії — завжди «станом на» дату прогнозу.

**Якість даних:** 768 замовлень без відгуків, 547 замовлень із кількома відгуками, 814 дублікатів `review_id`, 58,7 % відгуків без тексту, 610 товарів без категорії, 775 замовлень без позицій, 8 «delivered» без дати доставки та 6 скасованих, але доставлених.

**`customer_unique_id`** — це реальний покупець, а `customer_id` створюється на кожне замовлення, тож frequency і monetary за ним були б неправильними.

**Грануляність `seller_id × order_id`** потрібна, бо відгук належить замовленню. Без `DISTINCT` і попередньої агрегації одна оцінка рахувалася б стільки разів, скільки позицій продавця в замовленні, і рейтинг зміщувався б на користь великих кошиків.